In [2]:

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold, cross_val_score

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    LabelEncoder
)
from sklearn.impute import SimpleImputer

from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB


# ============================================================
# 2. LOAD DATASETS
# ============================================================

BASE_PATH = "/kaggle/input/competitions/mle-practice-test"

train = pd.read_csv(f"{BASE_PATH}/train.csv")
test = pd.read_csv(f"{BASE_PATH}/test.csv")
sample_submission = pd.read_csv(
    f"{BASE_PATH}/sample_submission.csv"
)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Submission shape:", sample_submission.shape)


# ============================================================
# 3. IDENTIFY TARGET COLUMN
# ============================================================

submission_columns = sample_submission.columns.tolist()

# Prefer the target column in sample submission
if len(submission_columns) >= 2:
    target_col = submission_columns[1]
else:
    raise ValueError("Invalid sample_submission format")

# Identify ID column
id_col = submission_columns[0]

# Verify target exists in training data
if target_col not in train.columns:

    possible_targets = [
        col for col in train.columns
        if col not in test.columns
    ]

    if len(possible_targets) == 1:
        target_col = possible_targets[0]
    else:
        raise ValueError(
            f"Could not identify target. "
            f"Possible columns: {possible_targets}"
        )

print("ID column:", id_col)
print("Target column:", target_col)


# ============================================================
# 4. INITIAL DATA INSPECTION
# ============================================================

print("\n--- TRAIN INFO ---")
print(train.info())

print("\n--- MISSING VALUES ---")
print(train.isnull().sum().sort_values(ascending=False))

print("\n--- TARGET DISTRIBUTION ---")
print(train[target_col].value_counts(dropna=False))

print("\n--- DUPLICATES ---")
print("Train duplicates:", train.duplicated().sum())
print("Test duplicates:", test.duplicated().sum())


# ============================================================
# 5. HANDLE MISSING TARGET VALUES
# ============================================================

# Rows without labels cannot be used for supervised learning
train = train.dropna(subset=[target_col]).copy()

# Reset index
train.reset_index(drop=True, inplace=True)
test.reset_index(drop=True, inplace=True)


# ============================================================
# 6. SEPARATE FEATURES AND TARGET
# ============================================================

# Exclude ID and target from training features
feature_cols = [
    col for col in train.columns
    if col not in [target_col, id_col]
]

# Keep only features that are also available in test
feature_cols = [
    col for col in feature_cols
    if col in test.columns
]

X = train[feature_cols].copy()
X_test = test[feature_cols].copy()

y = train[target_col].copy()

print("\nNumber of features:", len(feature_cols))


# ============================================================
# 7. HANDLE INFINITE VALUES
# ============================================================

X.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)


# ============================================================
# 8. IDENTIFY NUMERICAL AND CATEGORICAL COLUMNS
# ============================================================

numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)


# ============================================================
# 9. PREPROCESSING
# ============================================================

# Numerical:
# - Replace NaN with median
# - Standardize features

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical:
# - Replace NaN with most frequent value
# - One-hot encode categories
# - Handle unseen categories in test data

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)


# ============================================================
# 10. ENCODE TARGET LABELS
# ============================================================

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("\nTarget classes:", label_encoder.classes_)
print("Number of classes:", len(label_encoder.classes_))


# ============================================================
# 11. DEFINE 10 MACHINE LEARNING MODELS
# ============================================================

models = {

    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        C=1.0,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        n_jobs=-1,
        random_state=42
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        max_depth=None,
        n_jobs=-1,
        random_state=42
    ),

    "Hist Gradient Boosting": HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.08,
        max_leaf_nodes=31,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),

    "AdaBoost": AdaBoostClassifier(
        n_estimators=150,
        learning_rate=0.5,
        random_state=42
    ),

    "Support Vector Machine": SVC(
        C=1.0,
        kernel="rbf",
        random_state=42
    ),

    "K-Nearest Neighbors": KNeighborsClassifier(
        n_neighbors=5,
        weights="distance"
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=None,
        min_samples_split=2,
        random_state=42
    ),

    "Gaussian Naive Bayes": GaussianNB()

}


# ============================================================
# 12. STRATIFIED CROSS VALIDATION
# ============================================================

class_counts = np.bincount(y_encoded)
min_class_count = class_counts.min()

if min_class_count < 2:
    raise ValueError(
        "At least one target class has fewer than 2 samples. "
        "Stratified cross-validation is not possible."
    )

n_splits = min(5, min_class_count)

cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=42
)


# ============================================================
# 13. TRAIN AND EVALUATE ALL 10 MODELS
# ============================================================

results = []
trained_models = {}

for name, model in models.items():

    print("\n" + "=" * 60)
    print("Training:", name)

    pipeline = Pipeline(steps=[
        ("preprocessing", preprocessor),
        ("model", model)
    ])

    try:

        scores = cross_val_score(
            pipeline,
            X,
            y_encoded,
            cv=cv,
            scoring="accuracy",
            n_jobs=1,
            error_score="raise"
        )

        mean_score = scores.mean()
        std_score = scores.std()

        results.append({
            "Model": name,
            "CV Accuracy": mean_score,
            "Std": std_score
        })

        trained_models[name] = pipeline

        print("Fold scores:", np.round(scores, 4))
        print(f"Mean accuracy: {mean_score:.5f}")
        print(f"Std deviation: {std_score:.5f}")

    except Exception as e:

        print("Model failed:", name)
        print("Reason:", str(e))


# ============================================================
# 14. COMPARE ALL MODELS
# ============================================================

results_df = pd.DataFrame(results)

if results_df.empty:
    raise RuntimeError("All models failed.")

results_df = results_df.sort_values(
    by="CV Accuracy",
    ascending=False
).reset_index(drop=True)

print("\n\n========== MODEL COMPARISON ==========")
print(results_df.to_string(index=False))

print("\n========== BEST MODEL ==========")

best_model_name = results_df.iloc[0]["Model"]
best_accuracy = results_df.iloc[0]["CV Accuracy"]

print("Best Model:", best_model_name)
print("Best CV Accuracy:", round(best_accuracy, 5))


# ============================================================
# 15. TRAIN BEST MODEL ON FULL TRAINING DATA
# ============================================================

best_model = trained_models[best_model_name]

best_model.fit(X, y_encoded)

print("\nBest model trained on complete training dataset!")


# ============================================================
# 16. PREDICT TEST DATA
# ============================================================

test_predictions_encoded = best_model.predict(X_test)

# Convert encoded predictions to original labels
test_predictions = label_encoder.inverse_transform(
    test_predictions_encoded.astype(int)
)

print("\nPredictions completed!")
print("Total predictions:", len(test_predictions))



# ============================================================
# 17. CREATE FINAL SUBMISSION
# ============================================================

# Generate submission using all 800 test rows
submission = pd.DataFrame({
    "id": test["id"].values,
    "label": test_predictions
})

# Verify shape
print("Submission shape:", submission.shape)

# Verify columns
print("Submission columns:", submission.columns.tolist())

# Check missing values
print("Missing values:", submission.isnull().sum().sum())

# Check duplicate IDs
print("Duplicate IDs:", submission["id"].duplicated().sum())

# Validate predictions
assert len(submission) == len(test), "Row count mismatch!"
assert submission["label"].notna().all(), "Missing predictions!"
assert submission["id"].equals(test["id"]), "ID mismatch!"

# Save CSV
submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print("\n========== SUCCESS ==========")
print(submission.head(10))
print("\nTotal rows:", len(submission))
print("Total columns:", len(submission.columns))
print("Missing values:", submission.isnull().sum().sum())
print("\nFile saved successfully!")



Train shape: (3200, 35)
Test shape: (800, 34)
Submission shape: (249, 2)
ID column: id
Target column: label

--- TRAIN INFO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3200 entries, 0 to 3199
Data columns (total 35 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       3200 non-null   int64  
 1   age                      3200 non-null   int64  
 2   gender                   3200 non-null   object 
 3   bmi                      3200 non-null   float64
 4   smoking_status           3200 non-null   object 
 5   alcohol_consumption      2180 non-null   object 
 6   exercise_level           3200 non-null   object 
 7   diet_type                3200 non-null   object 
 8   sun_exposure             3200 non-null   object 
 9   income_level             3200 non-null   object 
 10  latitude_region          3200 non-null   object 
 11  vitamin_a_percent_rda    3200 non-null   float64
 12  vita